# Sentinel medical faithfulness classifier: RoBERTa + LoRA

This notebook trains and evaluates **only the medical answer-faithfulness classifier**. Sentinel's medical RAG/retrieval and LLM are already implemented and are not changed here. No Streamlit is used; keep React as the presentation choice.

Use a Colab GPU runtime (Runtime → Change runtime type → T4 GPU). You must provide a real, manually or reliably labeled **medical-only** CSV with `context`, `statement`, and `label` columns. Labels: `UNFAITHFUL`/`0` and `FAITHFUL`/`1`. Do not use the test set for threshold selection. No training can run until the labeled data is supplied.

In [ ]:
%pip -q install transformers==4.46.3 peft==0.13.2 accelerate==1.1.1 scikit-learn==1.5.2 pandas numpy matplotlib safetensors

In [ ]:
import os, random, json, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, auc)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments, EarlyStoppingCallback, set_seed)
from peft import LoraConfig, TaskType, get_peft_model

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Colab GPU runtime before training (T4 is supported).')
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)

LABEL_TO_ID = {'UNFAITHFUL': 0, 'FAITHFUL': 1}
ID_TO_LABEL = {value: key for key, value in LABEL_TO_ID.items()}
BASE_MODEL = 'roberta-base'
MAX_LENGTH = 512
OUTPUT_DIR = Path('/content/sentinel_roberta_lora_medical')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Optional: set this to a Drive/local Colab folder containing your trained DistilBERT checkpoint.
# It must include its tokenizer and model weights. Leave None if unavailable.
DISTILBERT_DIR = None

## Upload labeled medical examples

CSV columns: `context`, `statement`, `label`. Optional: `document_id`/`source_id` for group-aware splits, or a `split` column with `train`, `validation`, and `test`. Each example should pair retrieved medical context with an answer/statement and a trustworthy faithfulness label. Avoid putting near-duplicate answers or examples from one source across splits.

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
if len(csv_files) != 1:
    raise ValueError(f'Upload exactly one labeled medical CSV; found {csv_files}')
df = pd.read_csv(csv_files[0])
required = {'context', 'statement', 'label'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Missing required CSV columns: {sorted(missing)}')
df = df.dropna(subset=['context', 'statement', 'label']).copy()
df['context'] = df['context'].astype(str).str.strip()
df['statement'] = df['statement'].astype(str).str.strip()
df = df[df.context.ne('') & df.statement.ne('')].copy()
normalized = df['label'].astype(str).str.strip().str.upper().replace({'0':'UNFAITHFUL', '1':'FAITHFUL'})
unknown = sorted(set(normalized) - set(LABEL_TO_ID))
if unknown:
    raise ValueError(f'Labels must be UNFAITHFUL/FAITHFUL or 0/1; found {unknown}')
df['label_id'] = normalized.map(LABEL_TO_ID).astype(int)
if set(df.label_id.unique()) != {0, 1}:
    raise ValueError('The medical data must contain both labels.')
print('Examples:', len(df))
print('Label counts:', df.label_id.map(ID_TO_LABEL).value_counts().to_dict())
display(df[['context', 'statement', 'label_id']].head(3))

## Create train / validation / test splits

A provided `split` column is respected. Otherwise, examples are grouped by `document_id` or `source_id` when available so the same source cannot leak across splits. If neither source column exists, the fallback is stratified row splitting and may overstate performance if related examples are present. Ensure each split has both labels.

In [ ]:
if 'split' in df.columns:
    split_values = df['split'].astype(str).str.strip().str.lower()
    if not set(split_values).issubset({'train', 'validation', 'test'}):
        raise ValueError('split values must be train, validation, or test')
    train_df, val_df, test_df = [df.loc[split_values.eq(name)].copy() for name in ('train','validation','test')]
else:
    group_col = next((name for name in ('document_id', 'source_id') if name in df.columns), None)
    if group_col:
        if df[group_col].isna().any():
            raise ValueError(f'{group_col} contains empty source IDs')
        first = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
        train_idx, remainder_idx = next(first.split(df, df.label_id, groups=df[group_col]))
        remainder = df.iloc[remainder_idx]
        second = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
        val_rel, test_rel = next(second.split(remainder, remainder.label_id, groups=remainder[group_col]))
        train_df, val_df, test_df = df.iloc[train_idx].copy(), remainder.iloc[val_rel].copy(), remainder.iloc[test_rel].copy()
        print(f'Grouped split by {group_col}; verify enough distinct groups per class.')
    else:
        print('WARNING: no document/source ID; using stratified row splits. Related-example leakage may inflate metrics.')
        train_df, rest = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df.label_id)
        val_df, test_df = train_test_split(rest, test_size=0.5, random_state=SEED, stratify=rest.label_id)

for name, part in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    if set(part.label_id.unique()) != {0, 1}:
        raise ValueError(f'{name} split must contain both labels; counts={part.label_id.value_counts().to_dict()}')
    print(name, len(part), part.label_id.map(ID_TO_LABEL).value_counts().to_dict())

## Train RoBERTa-base with LoRA

The model receives the same pair format as Sentinel inference: `(context, statement)`, truncated to 512 tokens. Weighted cross-entropy addresses label imbalance. The best checkpoint is chosen by validation UNFAITHFUL F1; the test set is not used for model or threshold selection.

In [ ]:
from torch.utils.data import Dataset
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
class PairDataset(Dataset):
    def __init__(self, frame, with_labels=True):
        self.frame = frame.reset_index(drop=True)
        self.enc = tokenizer(self.frame.context.tolist(), self.frame.statement.tolist(), truncation=True, max_length=MAX_LENGTH)
        self.with_labels = with_labels
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        item = {key: values[i] for key, values in self.enc.items()}
        if self.with_labels: item['labels'] = int(self.frame.iloc[i].label_id)
        return item

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2, id2label=ID_TO_LABEL, label2id=LABEL_TO_ID,
    problem_type='single_label_classification')
model = get_peft_model(model, LoraConfig(
    task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=['query', 'value'], modules_to_save=['classifier']))
model.print_trainable_parameters()

counts = train_df.label_id.value_counts()
class_weights = torch.tensor([len(train_df)/(2*counts[0]), len(train_df)/(2*counts[1])], dtype=torch.float)
def compute_metrics(eval_pred):
    logits = eval_pred.predictions[0] if isinstance(eval_pred.predictions, tuple) else eval_pred.predictions
    predicted = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(eval_pred.label_ids, predicted),
            'unfaithful_f1': f1_score(eval_pred.label_ids, predicted, pos_label=0, zero_division=0),
            'unfaithful_recall': recall_score(eval_pred.label_ids, predicted, pos_label=0, zero_division=0)}

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = F.cross_entropy(outputs.logits, labels, weight=class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'), num_train_epochs=4,
    learning_rate=2e-4, per_device_train_batch_size=8, per_device_eval_batch_size=16,
    weight_decay=0.01, evaluation_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='unfaithful_f1', greater_is_better=True,
    save_total_limit=2, seed=SEED, fp16=True, report_to='none')
trainer = WeightedTrainer(
    model=model, args=args, train_dataset=PairDataset(train_df), eval_dataset=PairDataset(val_df),
    data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.train()
model = trainer.model.eval()
print('Best validation checkpoint:', trainer.state.best_model_checkpoint)

## Validation threshold, frozen test metrics, and AURC

`UNFAITHFUL=0`, so predictions are class 0 when its probability is at least the selected threshold. Threshold selection maximizes validation UNFAITHFUL F1 (recall is the first tie-breaker). It is then fixed before the single test evaluation. AURC uses test unfaithful risk among answers ranked by predicted faithfulness.

In [ ]:
from torch.utils.data import DataLoader
def probabilities(current_model, current_tokenizer, frame):
    loader = DataLoader(PairDataset(frame, with_labels=False), batch_size=16,
        collate_fn=DataCollatorWithPadding(current_tokenizer, return_tensors='pt'))
    current_model.eval(); all_logits = []
    with torch.inference_mode():
        for batch in loader:
            batch = {key: value.to(current_model.device) for key, value in batch.items()}
            all_logits.append(current_model(**batch).logits.cpu())
    return torch.softmax(torch.cat(all_logits), dim=-1).numpy()

def choose_threshold(labels, p_unfaithful):
    candidates = np.unique(np.r_[0.0, p_unfaithful, 1.0])
    rows = []
    for threshold in candidates:
        pred = np.where(p_unfaithful >= threshold, 0, 1)
        rows.append((f1_score(labels, pred, pos_label=0, zero_division=0),
                     recall_score(labels, pred, pos_label=0, zero_division=0),
                     accuracy_score(labels, pred), float(threshold)))
    return max(rows)[3]

def evaluate(labels, probs, threshold):
    p_unfaithful, p_faithful = probs[:, 0], probs[:, 1]
    pred = np.where(p_unfaithful >= threshold, 0, 1)
    order = np.argsort(-p_faithful, kind='stable')
    ordered_labels = labels[order]
    coverage = np.arange(1, len(labels)+1, dtype=float) / len(labels)
    risk = np.cumsum(ordered_labels == 0) / np.arange(1, len(labels)+1)
    return {'accuracy': float(accuracy_score(labels, pred)),
      'unfaithful_recall': float(recall_score(labels, pred, pos_label=0, zero_division=0)),
      'unfaithful_precision': float(precision_score(labels, pred, pos_label=0, zero_division=0)),
      'unfaithful_f1': float(f1_score(labels, pred, pos_label=0, zero_division=0)),
      'confusion_matrix_actual_rows_predicted_columns_labels_0_1': confusion_matrix(labels, pred, labels=[0,1]).tolist(),
      'unfaithful_pr_auc': float(average_precision_score(labels == 0, p_unfaithful)),
      'unfaithful_roc_auc': float(roc_auc_score(labels == 0, p_unfaithful)),
      'unfaithful_probability_threshold': float(threshold),
      'faithfulness_accept_threshold': float(1-threshold),
      'aurc': float(auc(np.r_[0.0, coverage], np.r_[0.0, risk])),
      'risk_coverage': {'coverage': coverage.tolist(), 'unfaithful_risk': risk.tolist()}}

val_probs = probabilities(model, tokenizer, val_df)
val_threshold = choose_threshold(val_df.label_id.to_numpy(), val_probs[:, 0])
val_report = evaluate(val_df.label_id.to_numpy(), val_probs, val_threshold)
test_probs = probabilities(model, tokenizer, test_df)
test_report = evaluate(test_df.label_id.to_numpy(), test_probs, val_threshold)
print('Validation-selected threshold and validation metrics:')
display(pd.Series({k:v for k,v in val_report.items() if k != 'risk_coverage'}).to_frame('value'))
print('Frozen test metrics:')
display(pd.Series({k:v for k,v in test_report.items() if k != 'risk_coverage'}).to_frame('value'))
import matplotlib.pyplot as plt
plt.plot(test_report['risk_coverage']['coverage'], test_report['risk_coverage']['unfaithful_risk'])
plt.xlabel('Coverage (fraction accepted)'); plt.ylabel('Unfaithful risk among accepted')
plt.title(f"Test risk-coverage; AURC={test_report['aurc']:.4f}"); plt.grid(True); plt.show()

## Optional: compare existing DistilBERT on the same split

Upload/mount the **trained** DistilBERT checkpoint and tokenizer, then set `DISTILBERT_DIR` above. A config/tokenizer without model weights is not a trained checkpoint. Both models use the same CSV examples and train/validation/test assignment; their thresholds are independently selected on validation. Compare test metrics and report measured results only—RoBERTa is not guaranteed to improve.

In [ ]:
comparison = {'RoBERTa-LoRA': test_report}
if DISTILBERT_DIR:
    baseline_tokenizer = AutoTokenizer.from_pretrained(DISTILBERT_DIR)
    baseline_model = AutoModelForSequenceClassification.from_pretrained(DISTILBERT_DIR).to('cuda').eval()
    baseline_val_probs = probabilities(baseline_model, baseline_tokenizer, val_df)
    baseline_threshold = choose_threshold(val_df.label_id.to_numpy(), baseline_val_probs[:, 0])
    baseline_test_probs = probabilities(baseline_model, baseline_tokenizer, test_df)
    comparison['DistilBERT'] = evaluate(test_df.label_id.to_numpy(), baseline_test_probs, baseline_threshold)
comparison_table = pd.DataFrame({name: {k:v for k,v in report.items() if k not in ('risk_coverage', 'confusion_matrix_actual_rows_predicted_columns_labels_0_1')} for name, report in comparison.items()}).T
display(comparison_table)

## Save artifacts for Sentinel

The saved LoRA adapter is not a standalone full model: keep its adapter config/weights and tokenizer together. `src/faithfulness.py` must load the adapter on top of its recorded `roberta-base` model, and Sentinel's acceptance threshold should come from the generated `threshold.json`. This notebook does not modify RAG, retrieval, LLM, or React.

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
report = {
  'domain': 'medical', 'base_model': BASE_MODEL, 'adapter': 'LoRA',
  'label_to_id': LABEL_TO_ID, 'split_sizes': {'train':len(train_df), 'validation':len(val_df), 'test':len(test_df)},
  'threshold_selection': 'maximum validation UNFAITHFUL F1; recall then accuracy tie-breaks',
  'validation': val_report, 'test': test_report,
  'comparison': {name: {k:v for k,v in result.items() if k != 'risk_coverage'} for name,result in comparison.items()}}
(OUTPUT_DIR / 'evaluation.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'threshold.json').write_text(json.dumps({'faithfulness_accept_threshold': 1-val_threshold}, indent=2), encoding='utf-8')
zip_path = '/content/sentinel_roberta_lora_medical.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file() and 'checkpoints' not in path.parts:
            archive.write(path, path.relative_to(OUTPUT_DIR.parent))
print('Saved adapter, tokenizer, evaluation.json, threshold.json:', zip_path)
from google.colab import files
files.download(zip_path)